|<h2>Book:</h2>|<h1><a href="https://open.substack.com/pub/mikexcohen/p/llm-breakdown-16-tokenization-words" target="_blank">50 ML projects to understand LLMs</a></h1>|
|-|:-:|
|<h2>Project:</h2>|<h1><b>[17] Probabilistic token selection</b></h1>|
|<h2>Author:<h2>|<h1>Mike X Cohen, <a href="https://sincxpress.com" target="_blank">sincxpress.com</a></h1>|

<br>

<i>Using the code without reading the book may lead to confusion or errors.</i>

In [1]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from transformers import AutoModelForCausalLM, AutoTokenizer

In [2]:
### matplotlib adjustments (commented lines are for dark mode)

# svg plots (higher-res)
import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

plt.rcParams.update({
    # 'figure.facecolor': '#282a2c',
    # 'figure.edgecolor': '#282a2c',
    # 'axes.facecolor':   '#282a2c',
    # 'axes.edgecolor':   '#DDE2F4',
    # 'axes.labelcolor':  '#DDE2F4',
    # 'xtick.color':      '#DDE2F4',
    # 'ytick.color':      '#DDE2F4',
    # 'text.color':       '#DDE2F4',
    'axes.spines.right': False,
    'axes.spines.top':   False,
    'axes.titleweight': 'bold',
    'axes.labelweight': 'bold',
    'savefig.dpi':50,
})

# **Part 1: PyTorch's multinomial function**

In [3]:
# a vector (must be tensor)
vect = torch.tensor([1,2,5],dtype=torch.float)

# sample a number
torch.multinomial(vect,1)

tensor([1])

In [4]:
# sample 10 times from that vector
vect[torch.multinomial(vect,10,replacement=True)]

tensor([1., 5., 5., 2., 1., 5., 2., 5., 5., 5.])

In [5]:
# 10k samples!
mn = torch.multinomial(vect,10000,replacement=True)

# collect the distribution
vals,counts = torch.unique(mn,return_counts=True)

# print the output values and how often they occurred
for v,c in zip(vals,counts):
  print(f'"{v}" was sampled {c} times ({c/len(mn):.2%})')

"0" was sampled 1297 times (12.97%)
"1" was sampled 2545 times (25.45%)
"2" was sampled 6158 times (61.58%)


In [6]:
# treat the vector as if it contains (scaled) probability values

print('Value | Observed | Expected')
print('------+----------+----------')

# again with more information
for v,c,vectval in zip(vals,counts,vect):

  observedFrequency = c/len(mn)
  expectedFrequency = vectval/torch.sum(vect)

  print(f' "{v}"  |  {observedFrequency:.2%}  |  {expectedFrequency:.2%}')

Value | Observed | Expected
------+----------+----------
 "0"  |  12.97%  |  12.50%
 "1"  |  25.45%  |  25.00%
 "2"  |  61.58%  |  62.50%


In [7]:
torch.multinomial??

# **Part 2: Common errors with torch.multinomial**

In [8]:
# # error 1: requires torch tensor
# torch.multinomial([1.,2,.3],1)
# torch.multinomial(np.array([1.,2,.3]),1)

In [9]:
# # error 2: default is no replacement
# torch.multinomial(vect,len(vect)+1)

In [10]:
# # error 3: only floats
# torch.multinomial(torch.tensor([1,1,1]),1)

In [11]:
# # error 4: only non-negative numbers
# torch.multinomial(torch.tensor([-1,1.,1]),1)

# **Part 3: Deterministic token selection**

In [12]:
# load pretrained GPT-2 model and tokenizer
llm = AutoModelForCausalLM.from_pretrained('gpt2')
tokenizer = AutoTokenizer.from_pretrained('gpt2')

llm.eval() # switch to eval mode

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [13]:
txt = "I don't want to grow"
tokens = tokenizer.encode(txt,return_tensors='pt')
tokens

tensor([[  40,  836,  470,  765,  284, 1663]])

In [14]:
# forward pass through the model
with torch.no_grad():
  outputs = llm(tokens)
outputs

CausalLMOutputWithCrossAttentions(loss=None, logits=tensor([[[ -39.3084,  -39.0100,  -41.8374,  ...,  -46.9337,  -44.9074,
           -39.5149],
         [-261.0096, -257.7495, -262.0256,  ..., -281.6742, -287.2552,
          -263.0469],
         [-168.8854, -167.9453, -176.2264,  ..., -181.3110, -180.2301,
          -172.5459],
         [ -55.5614,  -54.1664,  -60.2187,  ...,  -60.1240,  -64.3036,
           -57.6978],
         [-145.4407, -144.6209, -152.5711,  ..., -155.5938, -157.2356,
          -148.5867],
         [-104.6927, -104.4484, -112.1426,  ..., -108.8956, -115.2077,
          -108.4659]]]), past_key_values=DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer]), hidden_states=None, attentions=None, cross_attentions=None)

In [15]:
outputs.logits.shape

torch.Size([1, 6, 50257])

In [16]:
logits = outputs.logits[0,-1,:].detach()
logits_sm = F.softmax(logits,dim=-1)
logits.shape

torch.Size([50257])

In [17]:
# find the maximum
max_logit = logits.argmax()
print(f'The maximum logit is #{max_logit} with a softmax probability of {logits_sm[max_logit]:.2%}')
print(f'The max word is "{tokenizer.decode(max_logit)}"')

The maximum logit is #510 with a softmax probability of 33.87%
The max word is " up"


In [18]:
# plot the raw and softmax logits
_,axs = plt.subplots(1,2,figsize=(12,3))

axs[0].plot(max_logit,logits[max_logit],'rh')
axs[0].plot(logits,'gh',linewidth=.2,markerfacecolor=[.7,.9,.7,.3])
axs[0].set(xlabel='Token index',ylabel='Output logits (raw)',title='A) Raw output logits')

axs[1].plot(max_logit,logits_sm[max_logit],'rh')
axs[1].plot(logits_sm,'bh',linewidth=.2,markerfacecolor=[.7,.9,.7,.3])
axs[1].set(xlabel='Token index',ylabel='Softmax probability',title='B) Softmax probabilities')

plt.tight_layout()
plt.savefig('ch4_proj17_part3.png')
plt.show()

Output hidden; open in https://colab.research.google.com to view.

# **Part 4: Multinomial probabilistic selection**

In [19]:
for t in torch.multinomial(logits_sm,10,replacement=True):
  print(f'{txt}"{tokenizer.decode(t)}"\t({logits_sm[t]:.2%})')

I don't want to grow" to"	(1.04%)
I don't want to grow" up"	(33.87%)
I don't want to grow" crops"	(0.20%)
I don't want to grow" money"	(0.18%)
I don't want to grow" my"	(7.40%)
I don't want to grow" up"	(33.87%)
I don't want to grow" about"	(0.03%)
I don't want to grow" into"	(2.18%)
I don't want to grow" potatoes"	(0.14%)
I don't want to grow" as"	(0.47%)


# **Part 5: Top-k sampling**

In [20]:
k = 10
top_k = torch.topk(logits_sm,k)

print(txt,'___\n')

for i in range(k):
  val = top_k.values[i]
  tok = top_k.indices[i]
  print(f'{tok:5} ({val:5.1%}) is "{tokenizer.decode(tok)}"')

I don't want to grow ___

  510 (33.9%) is " up"
  616 ( 7.4%) is " my"
  257 ( 4.4%) is " a"
 1165 ( 2.3%) is " too"
  656 ( 2.2%) is " into"
  262 ( 2.1%) is " the"
  287 ( 2.1%) is " in"
  428 ( 1.7%) is " this"
  340 ( 1.6%) is " it"
 1468 ( 1.5%) is " old"


In [21]:
print(f'The top {k} options account for {logits_sm[top_k.indices].sum():.1%} of the probability mass.')

The top 10 options account for 59.1% of the probability mass.


In [22]:
# re-normalize to sum to 1
sm_top_k = F.softmax(logits[top_k.indices],dim=-1)

# what they look like:
print(sm_top_k)

# confirm: sum to 1
print(sm_top_k.sum())

tensor([0.5735, 0.1252, 0.0739, 0.0389, 0.0370, 0.0352, 0.0350, 0.0291, 0.0267,
        0.0255])
tensor(1.0000)


In [23]:
for t in torch.multinomial(sm_top_k,10):
  tidx = top_k.indices[t] # token index into vocab (not top-k)
  print(f'{txt}"{tokenizer.decode(tidx)}"\t({sm_top_k[t]:6.2%})')

I don't want to grow" the"	( 3.52%)
I don't want to grow" up"	(57.35%)
I don't want to grow" it"	( 2.67%)
I don't want to grow" in"	( 3.50%)
I don't want to grow" my"	(12.52%)
I don't want to grow" this"	( 2.91%)
I don't want to grow" into"	( 3.70%)
I don't want to grow" too"	( 3.89%)
I don't want to grow" a"	( 7.39%)
I don't want to grow" old"	( 2.55%)


# **Part 6: Top-p sampling**

In [24]:
# threshold
p_thresh = .9

# sort the probabilities
sorted_sm,sorted_idx = torch.sort(logits_sm,descending=True)

cumulative_probs = torch.cumsum(sorted_sm,dim=-1)
tokens2keep = torch.where(cumulative_probs>=p_thresh)[0][0]

# print the results
print(f'{tokens2keep} tokens accounts for {cumulative_probs[tokens2keep]:.1%} of the probability mass.')

282 tokens accounts for 90.0% of the probability mass.


In [25]:
tokens_idx = sorted_idx[:tokens2keep+1]

# renormalize from logits to probability
sm_top_p = F.softmax(logits[tokens_idx],dim=-1)

In [26]:
for t in torch.multinomial(sm_top_p,10):
  tidx = tokens_idx[t] # token index into vocab (not top-p)
  print(f'{txt}"{tokenizer.decode(tidx)}"\t({sm_top_p[t]:6.2%})')

I don't want to grow" into"	( 2.42%)
I don't want to grow" my"	( 8.22%)
I don't want to grow" as"	( 0.52%)
I don't want to grow" up"	(37.62%)
I don't want to grow" much"	( 0.11%)
I don't want to grow" old"	( 1.68%)
I don't want to grow" vegetables"	( 0.29%)
I don't want to grow" them"	( 0.43%)
I don't want to grow" anymore"	( 0.27%)
I don't want to grow" the"	( 2.31%)


# **Part 7: Chat with GPT2**

In [27]:
n_new_tokens = 15

# start from a prompt
txt = "I don't want to grow"
print(f'Start prompt: {txt}')

# loop over the new tokens
for _ in range(n_new_tokens):

  # tokenize, forward pass, get final logits
  tokens = tokenizer.encode(txt,return_tensors='pt')
  with torch.no_grad():
    logits = llm(tokens).logits[0,-1,:].detach()

  # softmax
  logits_sm = F.softmax(logits,dim=-1)

  # pick a new token
  newtok = torch.multinomial(logits_sm,1)

  # concatenate and print
  txt += tokenizer.decode(newtok)
  print(f'New token {_+1:2}: {txt}')

Start prompt: I don't want to grow
New token  1: I don't want to grow up
New token  2: I don't want to grow up,"
New token  3: I don't want to grow up," Clinton
New token  4: I don't want to grow up," Clinton said
New token  5: I don't want to grow up," Clinton said to
New token  6: I don't want to grow up," Clinton said to men
New token  7: I don't want to grow up," Clinton said to men who
New token  8: I don't want to grow up," Clinton said to men who questioned
New token  9: I don't want to grow up," Clinton said to men who questioned whether
New token 10: I don't want to grow up," Clinton said to men who questioned whether she
New token 11: I don't want to grow up," Clinton said to men who questioned whether she should
New token 12: I don't want to grow up," Clinton said to men who questioned whether she should ever
New token 13: I don't want to grow up," Clinton said to men who questioned whether she should ever speak
New token 14: I don't want to grow up," Clinton said to men who